# 東京23区・駅の縄張りランキング

歩行道路網から作る駅の範囲を、面積・推計人口・OSM登録POI数・人口加重平均道のりで見ます。POI記事の通し番号とは独立したテーマです。

[実行方法](../README.md) / [定義と限界](../docs/method.md) / [出典と公開条件](../docs/data-sources.md) / [検証](../docs/validation.md)

## 1. 準備
このテーマのディレクトリで依存を揃え、承認済みのローカル入力を取り込みます。`python -m station_territory.pipeline` は格子以降を再計算します。道路からの再計算は `python -m station_territory.replay_grid` です。このNotebookでは生成結果の整合を確かめて図表を読みます。自動ダウンロードは行いません。

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
from station_territory.territory import ROOT, RESULT, rank
from station_territory.provenance import verify_inputs
from station_territory.figures import METRICS

receipt = verify_inputs()
print("入力確認:", len(receipt["files"]), "ファイル")
stations = pd.read_parquet(RESULT / "all_stations.parquet")

## 2. 4種類の上位10駅
人口ありメッシュ面積は住宅用地ではありません。POIは店舗の実数ではなくOSM要素の登録数です。人口加重平均距離はメッシュ人口の面積按分による推計で、実際の駅利用を表しません。高齢化率と非人口加重の距離ランキングは扱いません。

In [ ]:
for field, (title, unit) in METRICS.items():
    eligible = stations.eligible & ((stations.population >= 1000) if field == "population_mean_m" else True)
    top, ties = rank(stations.assign(eligible=eligible), field)
    columns = ["rank", "name", field, "boundary_cut"]
    if field == "area_km2":
        columns.append("inhabited_mesh_area_km2")
    print(title, unit)
    display(top[columns])
    if len(ties) > len(top):
        print("10位同値を含む全件数:", len(ties))

## 3. 業種別POI
同じ物理施設に複数のOSM要素があっても統合していません。分類優先順と除外条件は固定設定を使います。

In [ ]:
policy = json.loads((ROOT / "configs/poi_policy.json").read_text(encoding="utf-8"))
for category in policy["priority"]:
    field = f"poi_{category}"
    top, ties = rank(stations, field)
    print(category)
    display(top[["rank", "name", field]])

## 4. 検証と地図
未割当人口を割当済みの駅に再配分しません。計算範囲やOSMの欠落は結果の限界です。隣県との重複比較が0件なら、その比較は未検証です。

In [ ]:
population = json.loads((RESULT / "population_validation.json").read_text(encoding="utf-8"))
territory = json.loads((RESULT / "validation.json").read_text(encoding="utf-8"))
print({key: population[key] for key in ["population_in_23", "assigned_population", "unassigned_population", "conservation_error_people"]})
print("ランキング対象駅:", territory["eligible_stations"])
print("隣県比較:", territory["neighbor_comparison"])
display(Image(filename=str(ROOT / "data/figures/territories.png")))

## 5. 記事での注意
駅範囲の広さやPOIの登録数は利便性・商業力の順位ではありません。道路は最大連結成分に限定し、駅に100m以内の接続先がない場合は上限なし最近傍接続を行う原実装を継承しています。図版署名は [masahiko.info](https://masahiko.info/) です。データ出典・公開条件は別に確認してください。生データ・実行済みNotebook・HTML・図表はGit管理外dataに保存します。